# MTRAG: BGE-M3 dense и sparse в Google Colab

Один проход BGE-M3 создаёт dense- и sparse-представления. Они сохраняются в независимые Elasticsearch-индексы и отдельные snapshot ZIP.

## 1. Зависимости

In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "FlagEmbedding==1.4.0",
    "elasticsearch>=9,<10",
])

0

## 2. Конфигурация

In [2]:
import base64
import json
import os
import shutil
import time
import zipfile
from itertools import islice
from pathlib import Path

import numpy as np
import requests
import torch
from elasticsearch import Elasticsearch, helpers
from FlagEmbedding.inference.embedder.encoder_only import m3 as bge_m3_module
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm

domains = ["clapnq", "cloud", "govt", "fiqa"]
model_name = "BAAI/bge-m3"
model_batch_size = 64
max_length = 512
encode_buffer_size = 128
bulk_size = 256

corpus_revision = "cc5b1d481b391181b89f7ced860308482e785463"
es_version = "9.3.3"
es_url = "http://127.0.0.1:9200"

workspace = Path("/content/mt-rag-bge-m3-hybrid")
data_dir = workspace / "corpora"
es_root = workspace / "elasticsearch"
es_home = es_root / f"elasticsearch-{es_version}"
es_archive = es_root / f"elasticsearch-{es_version}.tar.gz"
snapshot_root = workspace / "snapshots"
drive_output_dir = Path("/content/drive/MyDrive/mt-rag")
save_to_drive = True

bge_m3_module.tqdm = lambda iterable, *args, **kwargs: iterable
bge_m3_module.trange = lambda *args, **kwargs: range(*args)

for path in (workspace, data_dir, es_root, snapshot_root):
    path.mkdir(parents=True, exist_ok=True)
for mode in ("dense", "sparse"):
    for domain in domains:
        (snapshot_root / mode / domain).mkdir(parents=True, exist_ok=True)

## 3. Google Drive и GPU

In [3]:
if save_to_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_output_dir.mkdir(parents=True, exist_ok=True)

print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Tesla T4
14.6 GB


## 4. Корпусы

In [4]:
corpus_url = (
    "https://raw.githubusercontent.com/IBM/mt-rag-benchmark/"
    f"{corpus_revision}/corpora/passage_level/{{domain}}.jsonl.zip"
)


def corpus_path(domain):
    return data_dir / f"{domain}.jsonl.zip"


def download(url, destination):
    if destination.exists():
        return
    with requests.get(url, stream=True) as response:
        response.raise_for_status()
        with destination.open("wb") as output:
            shutil.copyfileobj(response.raw, output)


corpus_sizes = {}
for domain in domains:
    path = corpus_path(domain)
    download(corpus_url.format(domain=domain), path)
    with zipfile.ZipFile(path) as archive:
        with archive.open(f"{domain}.jsonl") as stream:
            corpus_sizes[domain] = sum(1 for _ in stream)
    print(domain, corpus_sizes[domain])

clapnq 183408
cloud 72442
govt 49607
fiqa 61022


## 5. Чтение JSONL

In [5]:
def iter_passages(domain):
    with zipfile.ZipFile(corpus_path(domain)) as archive:
        with archive.open(f"{domain}.jsonl") as stream:
            for row_idx, line in enumerate(stream):
                row = json.loads(line)
                yield {
                    "row_idx": row_idx,
                    "doc_id": str(row["id"]),
                    "title": row.get("title", ""),
                    "text": row["text"],
                    "url": row.get("url", ""),
                }


def batched(rows, size):
    iterator = iter(rows)
    while batch := list(islice(iterator, size)):
        yield batch

## 6. Elasticsearch

In [6]:
server_url = (
    f"https://artifacts.elastic.co/downloads/elasticsearch/"
    f"elasticsearch-{es_version}-linux-x86_64.tar.gz"
)

if not es_home.exists():
    download(server_url, es_archive)
    shutil.unpack_archive(es_archive, es_root)
    es_archive.unlink()


def elasticsearch_ready():
    try:
        return requests.get(es_url, timeout=2).ok
    except requests.RequestException:
        return False


if not elasticsearch_ready():
    subprocess.run(
        ["useradd", "-m", "esuser"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    subprocess.run(
        ["chown", "-R", "esuser:esuser", str(es_root), str(snapshot_root)],
        check=True,
    )
    command = [
        "runuser",
        "-u",
        "esuser",
        "--",
        str(es_home / "bin/elasticsearch"),
        "-Ediscovery.type=single-node",
        "-Expack.security.enabled=false",
        "-Expack.ml.enabled=false",
        "-Ehttp.host=127.0.0.1",
        f"-Epath.repo={snapshot_root}",
    ]
    environment = os.environ.copy()
    environment["ES_JAVA_OPTS"] = "-Xms3g -Xmx3g"
    log = open(es_root / "elasticsearch.log", "w")
    es_process = subprocess.Popen(
        command,
        env=environment,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    while not elasticsearch_ready():
        if es_process.poll() is not None:
            raise RuntimeError((es_root / "elasticsearch.log").read_text())
        time.sleep(2)

es = Elasticsearch(
    es_url,
    request_timeout=300,
    retry_on_timeout=True,
    retry_on_status=(429, 502, 503, 504),
    max_retries=8,
)
print(es.info()["version"]["number"])

9.3.3


## 7. BGE-M3

In [7]:
model = BGEM3FlagModel(
    model_name,
    devices="cuda:0",
    use_fp16=True,
)


def dense_to_base64(vector):
    data = np.asarray(vector, dtype=">f4").tobytes()
    return base64.b64encode(data).decode("ascii")


def sparse_to_dict(vector):
    return {
        str(token_id): float(weight)
        for token_id, weight in dict(vector).items()
        if float(weight) > 0
    }


def encode(texts):
    output = model.encode_corpus(
        texts,
        batch_size=model_batch_size,
        max_length=max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense = np.asarray(output["dense_vecs"], dtype=np.float32)
    dense /= np.maximum(np.linalg.norm(dense, axis=1, keepdims=True), 1e-12)
    sparse = [sparse_to_dict(row) for row in output["lexical_weights"]]
    return dense, sparse

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 8. Dense- и sparse-индексы

In [8]:
def index_name(domain, mode):
    return f"mtrag-{domain}-bge-m3-{mode}"


def create_index(domain, mode):
    name = index_name(domain, mode)
    if es.indices.exists(index=name):
        return name
    if mode == "dense":
        embedding = {
            "type": "dense_vector",
            "dims": 1024,
            "similarity": "dot_product",
            "index": True,
            "index_options": {"type": "int8_hnsw"},
        }
    else:
        embedding = {
            "type": "sparse_vector",
            "index_options": {"prune": False},
        }
    es.indices.create(
        index=name,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "-1",
            "index.mapping.exclude_source_vectors": True,
        },
        mappings={
            "dynamic": "strict",
            "properties": {
                "row_idx": {"type": "integer"},
                "doc_id": {"type": "keyword"},
                "title": {"type": "text"},
                "text": {"type": "text"},
                "url": {"type": "keyword", "ignore_above": 2048},
                "embedding": embedding,
            },
        },
    )
    return name


def existing_ids(name):
    es.indices.refresh(index=name)
    return {
        hit["_id"]
        for hit in helpers.scan(
            es,
            index=name,
            query={"query": {"match_all": {}}},
            source=False,
            size=2000,
            request_timeout=900,
        )
    }

## 9. Индексация

In [9]:
def index_domain(domain):
    dense_name = create_index(domain, "dense")
    sparse_name = create_index(domain, "sparse")
    dense_ids = existing_ids(dense_name)
    sparse_ids = existing_ids(sparse_name)
    rows = (
        row
        for row in iter_passages(domain)
        if row["doc_id"] not in dense_ids or row["doc_id"] not in sparse_ids
    )

    with tqdm(
        total=corpus_sizes[domain],
        initial=len(dense_ids & sparse_ids),
        desc=domain,
    ) as progress:
        for batch in batched(rows, encode_buffer_size):
            dense_vectors, sparse_vectors = encode([row["text"] for row in batch])
            actions = []
            for row, dense, sparse in zip(batch, dense_vectors, sparse_vectors):
                if row["doc_id"] not in dense_ids:
                    actions.append({
                        "_op_type": "index",
                        "_index": dense_name,
                        "_id": row["doc_id"],
                        "_source": {**row, "embedding": dense_to_base64(dense)},
                    })
                if row["doc_id"] not in sparse_ids:
                    actions.append({
                        "_op_type": "index",
                        "_index": sparse_name,
                        "_id": row["doc_id"],
                        "_source": {**row, "embedding": sparse},
                    })
            helpers.bulk(
                es,
                actions,
                chunk_size=bulk_size,
                max_chunk_bytes=32 * 1024 * 1024,
                max_retries=5,
                retry_on_status=(429,),
                request_timeout=900,
            )
            progress.update(len(batch))

    es.indices.put_settings(
        index=f"{dense_name},{sparse_name}",
        settings={"index": {"refresh_interval": "1s"}},
    )
    es.indices.refresh(index=f"{dense_name},{sparse_name}")
    print(dense_name, es.count(index=dense_name)["count"])
    print(sparse_name, es.count(index=sparse_name)["count"])

## 10. Snapshot ZIP

In [10]:
def es_request(method, path, body=None, **kwargs):
    response = requests.request(method, es_url + path, json=body, **kwargs)
    response.raise_for_status()
    return response.json() if response.content else None


def pack_directory(source, destination):
    with zipfile.ZipFile(
        destination,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:
        for path in source.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(source))
    return destination


def snapshot_state(repository, snapshot):
    response = requests.get(
        f"{es_url}/_snapshot/{repository}/{snapshot}",
        timeout=30,
    )
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()["snapshots"][0]["state"]


def save_snapshot(domain, mode):
    repository = f"mtrag-bge-m3-{mode}-{domain}-repository"
    snapshot = f"mtrag-bge-m3-{mode}-{domain}"
    repository_dir = snapshot_root / mode / domain
    repository_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["chown", "-R", "esuser:esuser", str(repository_dir)],
        check=True,
    )

    es_request(
        "PUT",
        f"/_snapshot/{repository}",
        {"type": "fs", "settings": {"location": str(repository_dir)}},
    )

    snapshot_path = f"/_snapshot/{repository}/{snapshot}"
    state = snapshot_state(repository, snapshot)
    while state == "IN_PROGRESS":
        time.sleep(5)
        state = snapshot_state(repository, snapshot)
    if state not in (None, "SUCCESS"):
        es_request("DELETE", snapshot_path)
        state = None
    if state is None:
        es_request(
            "PUT",
            snapshot_path,
            {
                "indices": index_name(domain, mode),
                "include_global_state": False,
            },
            params={"wait_for_completion": "true"},
            timeout=3600,
        )

    es_request("DELETE", f"/_snapshot/{repository}")
    archive = workspace / f"mtrag_bge_m3_{mode}_{domain}.zip"
    pack_directory(repository_dir, archive)

    if save_to_drive:
        shutil.copy2(archive, drive_output_dir / archive.name)

    print(archive)

## 11. Запуск

In [11]:
for domain in domains:
    index_domain(domain)
    save_snapshot(domain, "dense")
    save_snapshot(domain, "sparse")

clapnq:   0%|          | 0/183408 [00:00<?, ?it/s]

/tmp/ipykernel_612/668215238.py:35: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  helpers.bulk(


mtrag-clapnq-bge-m3-dense 183408
mtrag-clapnq-bge-m3-sparse 183408
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_dense_clapnq.zip
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_sparse_clapnq.zip


cloud:   0%|          | 0/72442 [00:00<?, ?it/s]

/tmp/ipykernel_612/668215238.py:35: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  helpers.bulk(


...

/tmp/ipykernel_612/668215238.py:35: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  helpers.bulk(


mtrag-govt-bge-m3-dense 49607
mtrag-govt-bge-m3-sparse 49607
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_dense_govt.zip
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_sparse_govt.zip


fiqa:   0%|          | 0/61022 [00:00<?, ?it/s]

/tmp/ipykernel_612/668215238.py:35: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  helpers.bulk(


mtrag-fiqa-bge-m3-dense 61022
mtrag-fiqa-bge-m3-sparse 61022
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_dense_fiqa.zip
/content/mt-rag-bge-m3-hybrid/mtrag_bge_m3_sparse_fiqa.zip
